### Import important library

In [1]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import yaml
from pathlib import Path
import time
from collections import defaultdict
from google import genai

In [2]:
from functions.utils.logging import get_logger
from functions.utils.config  import PROJECT_ROOT, load_config
from functions.utils.llm_client import build_llm_client_from_yaml
# from functions.utils.text_embeddings import GoogleEmbeddingModel
from functions.core.context_builder import build_user_context
from functions.core.history import build_history_summary

#### Replace from functions.utils.text_embeddings import GoogleEmbeddingModel -> embed_texts_gemini

In [3]:
def embed_texts_gemini(texts:list[str],output_dim:int=768,task_type: str = "RETRIEVAL_DOCUMENT")->np.ndarray:
    if not texts:
        return np.zeros((0,output_dim or 0), dtype=np.float32)
    # step01 : API key
    key = os.getenv("GOOGLE_API_KEY")
    if not key:
        raise ValueError("GOOGLE_API_KEY is not set")
    # step02 : client creation
    client = genai.Client(api_key=key)
    vectors = []
    for raw in texts:
        text = raw.strip() or " "
        resp = client.models.embed_content(
            model="gemini-embedding-001",
            contents=text,
            config={"task_type": task_type},
        )
        if not hasattr(resp, "embeddings") or not resp.embeddings:
            raise RuntimeError("Invalid embedding response")
        values = resp.embeddings[0].values
        vectors.append(values)
    # step03 : ndarray + float32 (friend logic)
    mat = np.asarray(vectors, dtype=np.float32)
    # step04 : optional truncation (before normalize)
    if output_dim is not None:
        if output_dim > mat.shape[1]:
            raise ValueError(
                f"output_dim {output_dim} > embedding dim {mat.shape[1]}"
            )
        mat = mat[:, :output_dim]
    # step05 : L2 normalize (friend logic)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms = np.where(norms == 0.0, 1.0, norms)
    mat = mat / norms

    return mat
texts = [
    "Hello, world!",
    "Vertex AI is great for embeddings.",
    "I hate everything about you",
    "21 gungs"
]

vectors = embed_texts_gemini(
    texts,
    output_dim=768,
)

print(vectors.shape)      # (2, 768)
print(vectors.dtype)      # float32
print(np.linalg.norm(vectors[0]))  # ~1.0


(4, 768)
float32
1.0


In [4]:
vectors

array([[-0.03438904,  0.00846276,  0.02816775, ...,  0.01759785,
         0.03002233, -0.00213392],
       [-0.01362397,  0.03445741, -0.02554202, ...,  0.04793327,
         0.01554968, -0.00532724],
       [-0.0131421 , -0.00189351,  0.0002318 , ...,  0.03461472,
         0.04891067,  0.03601924],
       [-0.03315475,  0.0152856 ,  0.001271  , ..., -0.04313601,
         0.01114013,  0.00477143]], dtype=float32)

<hr>

### QueryData

In [5]:
from google.cloud import bigquery

In [6]:
class DataQuery:
    def __init__(self):
        self.client = bigquery.Client()
    def get_students(self):
        query = """
            SELECT *
            FROM `poc-piloturl-nonprod.gold_layer.students`
        """
        df = self.client.query(query).to_dataframe()
        return df
    def get_interactions(self):
        query = """
            SELECT *
            FROM `poc-piloturl-nonprod.gold_layer.interactions`
        """
        df = self.client.query(query).to_dataframe()
        return df 
    def get_user_events_json(self):
        query = """
        SELECT *
        FROM `poc-piloturl-nonprod.gold_layer.feeds`
        """
        df = self.client.query(query).to_dataframe()
        # ensure created_at is ISO-8601 Z format
        df["created_at"] = df["created_at"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

        feeds_lookup: Dict[str, Dict[str, Any]] = {}
        for _,row in df.iterrows():
            feed_id = row["feed_id"]
            feeds_lookup[feed_id] = {
                "feed_id"        : feed_id,
                "title"          : row["title"],
                "feed_text"      : row["feed_text"],
                "tags"           : row["tags"],                     
                "language"       : row["language"],
                "created_at"     : row["created_at"],
                "source"         : row["source"],
                "url"            : row["url"],
                "views"          : int(row["views"]),
                "embedding_input": row["embedding_input"]
            }
        return feeds_lookup
# dq = DataQuery()
# dq.get_students()   

### Cloud storage

In [7]:
import json
import numpy as np
import io
from datetime import datetime, timedelta, timezone
from google.cloud import storage

class GoogleCloudStorage:
    def __init__(self,bucket_name):
        self.client = storage.Client()
        try:
            self.bucket = self.client.get_bucket(bucket_name)
            print(f"Bucket exists  : {bucket_name}")
        except Exception:
            self.bucket = self.client.create_bucket(bucket_name, location=location)
            print(f"Bucket created : {bucket_name}")
            
    def blob_exists(self, blob_path) -> bool:
        '''check if object exists'''
        return self.bucket.blob(blob_path).exists()

    ### ---------- Upload folder function ----------- ###
    def upload_json(self,blob_path,json_data):
        '''upload json file to bucket'''
        blob   = self.bucket.blob(blob_path)
        
        blob.upload_from_string(
            json.dumps(json_data,ensure_ascii = False),
            content_type = "application/json"
        )
        print(f"uploaded JSON -> gs://{self.bucket.name}/{blob_path}")

    def upload_text(self, blob_path, text_data):
        '''upload text file to bucket'''
        blob   = self.bucket.blob(blob_path)

        blob.upload_from_string(
            text_data,
            content_type = "text/plain"
        )
        print(f"Uploaded text -> gs://{self.bucket.name}/{blob_path}")

    def upload_npy(self, blob_path, array):
        '''upload embedding vector'''
        buffer = io.BytesIO()
        np.save(buffer, array)
        buffer.seek(0)
        
        blob = self.bucket.blob(blob_path)
        blob.upload_from_file(
            buffer,
            content_type = "application/octet-stream"
        )
        print(f"Uploaded NPY -> gs://{self.bucket.name}/{blob_path}")
        
    ### ---------- Read file function ----------- ###
    def read_json(self, blob_path):
        '''read json file'''
        blob   = self.bucket.blob(blob_path)
        return json.loads(blob.download_as_text())

    def read_text(self, blob_path):
        '''read text file'''
        blob   = self.bucket.blob(blob_path)
        return blob.download_as_text()

    def read_npy(self, blob_path):
        '''read .npy (embedding vector) file'''
        blob   = self.bucket.blob(blob_path)

        buffer = io.BytesIO()
        blob.download_to_file(buffer)
        buffer.seek(0)
        return np.load(buffer)
        
    ### ---------- Creation folder function ----------- ###
    def create_folder(self,folder_path):
        '''Creating folder and sub folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blob = self.bucket.blob(folder_path)
        blob.upload_from_string("")
        print(f"Folder created : gs://{self.bucket}/{folder_path}")
        
    ### ---------- Remove function ----------- ###
    def delete_blob(self, blob_path):
        blob   = self.bucket.blob(blob_path)
        if blob.exists():
            blob.delete()
        print(f"Deleted: gs://{self.bucket_name}/{blob_path}")

    def delete_folder(self, folder_path):
        '''Remove nest blob(file) in folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blobs = self.bucket.list_blobs(prefix=folder_path)
        count = 0
        for blob in blobs:
            blob.delete()
            count += 1
    
        print(f"Deleted {count} objects under gs://{self.bucket_name}/{folder_path}")

    # def delete_by_ttl(self, prefix, ttl: timedelta):
    #     '''Remove folder with setting time
    #     timeformat support
    #     timedelta(
    #         days=...,
    #         seconds=...,
    #         microseconds=...,
    #         milliseconds=...,
    #         minutes=...,
    #         hours=...,
    #         weeks=...
    #     )
    #     '''
    #     now    = datetime.now(timezone.utc)
    #     blob   = self.bucket.blob(blob_path)
    #     deleted = 0
    #     for blob in blobs:
    #         if blob.time_created and now - blob.time_created > ttl:
    #             blob.delete()
    #             deleted += 1
    #     print(f"TTL cleanup deleted {deleted} objects under {prefix}")
        
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake-feeds")

Bucket exists  : hyde-datalake-feeds


### Helper function

In [8]:
def ensure_dir(path: str) -> None:
    """Create directory if it does not exist (idempotent)."""
    os.makedirs(path, exist_ok=True)
    
def _read_hyde_config(cfg: Dict[str, Any]) -> Tuple[int, int, int, bool, str]:
    """
    Read HyDE-related configuration with safe defaults.

    Returns
    -------
    history_threshold:
        Event count threshold for prompt selection
    recent_k:
        Max number of recent feeds used in HistorySummary
    feed_text_max_chars:
        Per-feed text truncation limit
    include_recent_feeds:
        Whether HistorySummary may include feed snippets
    query_embedding_model_name:
        Embedding model for HyDE queries
    """
    hyde_cfg = cfg.get("hyde", {}) if isinstance(cfg, dict) else {}

    history_threshold = int(hyde_cfg.get("history_threshold", 5))
    recent_k = int(hyde_cfg.get("recent_k", 5))
    feed_text_max_chars = int(hyde_cfg.get("feed_text_max_chars", 240))
    include_recent_feeds = bool(hyde_cfg.get("include_recent_feeds", True))

    # Default to same embedding family as feed embeddings
    query_embedding_model_name = str(
        hyde_cfg.get("query_embedding_model_name")
        or cfg.get("embeddings", {}).get("model_name", "")
        or "gemini-embedding-001"
    )

    # Hard safety guards
    history_threshold = max(1, history_threshold)
    recent_k = max(0, min(recent_k, 10))
    feed_text_max_chars = max(0, min(feed_text_max_chars, 2000))

    return (
        history_threshold,
        recent_k,
        feed_text_max_chars,
        include_recent_feeds,
        query_embedding_model_name,
    )
    
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    """
    Deterministic JSONL reader.

    Order is preserved, which is critical for any downstream alignment.
    """
    rows: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f"Invalid JSONL at line {line_no}: {e}") from e
    return rows

def load_prompts() -> Dict[str, str]:
    """
    Load HyDE prompt templates from parameters/prompts.yaml.

    Expected structure:
      hyde_prompts:
        hyde_a: "..."
        hyde_b: "..."
        hyde_c: "..."
    """
    import yaml

    prompts_path = PROJECT_ROOT / "parameters" / "prompts.yaml"
    with prompts_path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}

    return data.get("hyde_prompts", {}) or {}

# =============================================================================
# Prompt selection and rendering
# =============================================================================
def choose_hyde_prompt_key(num_events: int, history_threshold: int = 5) -> str:
    """
    Select HyDE prompt variant based on interaction volume.

    Rules
    -----
    - num_events >= history_threshold → history-heavy (hyde_b)
    - num_events <= 1               → onboarding / sparse (hyde_c)
    - otherwise                     → mixed (hyde_a)
    """
    if num_events >= history_threshold:
        return "hyde_b"
    if num_events <= 1:
        return "hyde_c"
    return "hyde_a"


def render_prompt(
    template: str,
    preferred_language: str,
    user_context_text: str,
    history_summary_text: Optional[str],
) -> str:
    """
    Render a prompt template using strict placeholder substitution.

    Supported placeholders:
    - {{preferred_language}}
    - {{UserContextText}}
    - {{HistorySummaryText}}

    No templating engine is used on purpose to keep behavior explicit.
    """
    s = template.replace("{{preferred_language}}", preferred_language or "th")
    s = s.replace("{{UserContextText}}", user_context_text or "")
    s = s.replace("{{HistorySummaryText}}", history_summary_text or "")
    return s

# =============================================================================
# HyDE output handling
# =============================================================================
def _extract_hyde_query_texts(hyde_json: Dict[str, Any]) -> List[str]:
    """
    Extract query_text values from HyDE JSON output.

    Expected structure:
      {
        "hyde_queries": [
          {"query_id": "...", "query_text": "...", ...},
          ...
        ]
      }

    Order is preserved and MUST match embedding row order.
    """
    if not isinstance(hyde_json, dict):
        raise ValueError("hyde_output must be a dict")

    items = hyde_json.get("hyde_queries") or []
    if not isinstance(items, list):
        raise ValueError("hyde_output.hyde_queries must be a list")

    out: List[str] = []
    for i, it in enumerate(items):
        if not isinstance(it, dict):
            raise ValueError(f"hyde_output.hyde_queries[{i}] must be an object")
        out.append(str(it.get("query_text") or "").strip())

    return out


def _l2_normalize_rows(x: np.ndarray) -> np.ndarray:
    """
    Row-wise L2 normalization.

    Zero rows are left as zero to avoid NaNs.
    """
    if x.ndim != 2:
        raise ValueError("Expected 2D array for row normalization")

    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    return (x / norms).astype(np.float32)


def _atomic_save_npy(path: str, arr: np.ndarray) -> None:
    """
    Best-effort atomic .npy write.

    Writes to a temp file and renames to avoid partial reads.
    """
    tmp = path + ".tmp.npy"
    np.save(tmp, arr)
    os.replace(tmp, path)

# Main

### Load resource

In [9]:
cfg = load_config()
out_dir = cfg["artifacts"]["user_query_bundles_dir"]
verbose = 1
bq = DataQuery()

In [10]:
students = bq.get_students() 
interactions = bq.get_interactions() 
feeds_lookup = bq.get_user_events_json()

/usr/local/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


### Read HyDE-related configuration once

In [11]:
(history_threshold,recent_k,feed_text_max_chars,include_recent_feeds,query_embedding_model_name) = _read_hyde_config(cfg)
expected_dim = int(cfg.get("embeddings", {}).get("dim", 0) or 0)

In [12]:
prompts = load_prompts()
if not prompts:
    raise ValueError("hyde_prompts missing from parameters/prompts.yaml")
client = build_llm_client_from_yaml(
    parameters_path=str(PROJECT_ROOT / "parameters" / "parameters.yaml"),
    credentials_path=str(PROJECT_ROOT / "parameters" / "credentials.yaml"),
)
# query_embedder = GoogleEmbeddingModel(
#     model_name=query_embedding_model_name,
#     credentials_path=str(PROJECT_ROOT / "parameters" / "credentials.yaml"),
# )
now_iso = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

In [13]:
verbose = 1

In [14]:
# ------------------------------------------------------------------
# Generate one cached bundle per student
# ------------------------------------------------------------------
rows = []
logger = get_logger("pipeline_1_user_hyde")
for _, row in students.iterrows():
    student_row = row.to_dict()     # convert pd -> dict for each row
    student_id  = str(student_row.get("student_id","")).strip()
    # if student_id != "stu_p001":
    #     continue
    if not student_id or student_id.lower() == "nan":
        raise ValueError(f"Invalid student_id in students.csv: {student_row!r}")
    
    user_ctx = build_user_context(student_row)
    pref_lang = user_ctx.user_context_json.get("preferred_language","th")

    user_events = interactions[interactions["user_id"] == student_id]   # <- user event from interaction.csv
    num_events  = int(len(user_events))

    history_summary_text : Optional[str] = None
    #** Crate by combe data for each person student **#
    if num_events > 0:
        history_summary_text = build_history_summary(
            user_events,
            preferred_language   = pref_lang,
            include_recent_feeds = include_recent_feeds,
            recent_k             = recent_k,
            feeds_lookup         = feeds_lookup or None,
            feed_text_max_chars  = feed_text_max_chars,
        )
    
    prompt_key = choose_hyde_prompt_key(num_events,history_threshold)
    template = prompts.get(prompt_key)
    if not template:
        raise ValueError(f"Missing prompt '{prompt_key}' in pormpts.yaml")
    prompt = render_prompt(
            template=template,
            preferred_language=pref_lang,
            user_context_text=user_ctx.user_context_text,
            history_summary_text=history_summary_text,
        )
    # ------------------------------------------------------------------
    # LLM call (JSON-only)
    # ------------------------------------------------------------------
    hyde_json = client.generate_json(prompt)
    # ------------------------------------------------------------------
    # Embed HyDE queries for fast serving
    # ------------------------------------------------------------------
    hyde_query_texts = _extract_hyde_query_texts(hyde_json)

    # if hyde_query_texts:
    #     emb = query_embedder.embed_documents(hyde_query_texts)
    #     print("#"*100)
    #     print(f"emb.shape ->\n{emb.shape}")
    #     emb = np.asarray(emb, dtype = np.float32)
    #     if emb.ndim != 2:
    #         raise ValueError(f"Invalid embedding shape {emb.shape}")
    #     emb = _l2_normalize_rows(emb)
    #     if expected_dim and emb.shape[1] != expected_dim:
    #         raise ValueError(
    #             f"Embedding dim mismatch for student = {student_id}:"
    #             f"got {emb.shape[1]} expected {expected_dim}"
    #         )
    #     dim = int(emb.shape[1])
    # else:
    #     dim = expected_dim or 0
    #     emb = np.zeros((0,dim), dtype=np.float32)
    if hyde_query_texts:
        emb = embed_texts_gemini(
            texts=hyde_query_texts,
            output_dim=768,
            task_type="RETRIEVAL_DOCUMENT",
        )
        if emb.ndim != 2:
            raise ValueError(f"Invalid embedding shape {emb.shape}")
        dim = int(emb.shape[1])
        print("#" * 100)
        print(f"emb.shape -> {emb.shape}")
    else:
        dim = expected_dim or 0
        emb = np.zeros((0, dim), dtype=np.float32)
    print("*"*50)
    print(emb.shape)

    emb_filename = f"{student_id}_hyde_q_emb.npy"
    emb_path     = os.path.join(out_dir, emb_filename)
    _atomic_save_npy(emb_path, emb)
    # ---------------------------------------------------
    # Persist cached bundle for online serving
    # ---------------------------------------------------
    bundle: Dict[str, Any] = {
        "bundle_version"        : "v2_hyde_embedded_queries",
        "student_id"            : student_id,
        "generated_at"          : now_iso,
        "prompt_key"            : prompt_key,
        "preferred_language"    : pref_lang,
        "num_events"            : num_events,
        "user_context_json"     : user_ctx.user_context_json,
        "user_context_text"     : user_ctx.user_context_text,
        "history_summary_text"  : history_summary_text,
        "hyde_output"           : hyde_json,
        "hyde_query_embeddings" : {
            "path"        : emb_filename,
            "model"       : query_embedding_model_name,
            "dim"         : dim,
            "dtype"       : "float32",
            "num_queries" : int(len(hyde_query_texts)),
            "normalized"  : True,
        },
    }

    out_path = os.path.join(out_dir, f"{student_id}.json")
    with open(out_path,"w",encoding="utf-8") as f:
        json.dump(bundle,f,ensure_ascii=False,indent=2)
    logger.info(
        "wrote HyDE bundle student_id=%s events=%d prompt=%s",
        student_id,
        num_events,
        prompt_key,
    )
    if verbose > 0:
        print(_)
        print(f"student_row -> \n {student_row}")
        print(f"user_ctx -> \n {user_ctx}")
        print(f"user_events -> \n {user_events}")
        print(f"promt_key -> {prompt_key}")
        print(f"hyde_query_texts->{hyde_query_texts}")
        print(f"emb -> \n{emb}")
        print(f"emb_path ->\n{emb_path}")
        print(f"bundle->\n{bundle}")
        print("#"*100)
    # for vec in emb:  # emb.shape = (N, D)
    #     rows.append({
    #         "user_id": student_row["student_id"],
    #         "created_at": datetime.now(timezone.utc).isoformat(),
    #         # "embedding": vec.tolist()
    #         "embedding": "x"
    
    #     })
    # Ingest to GCS
    cgs.create_folder(
        folder_path = f"{student_id}/embedding/"
    )
    cgs.create_folder(
        folder_path = f"{student_id}/hyde/"
    )
    cgs.create_folder(
        folder_path = f"{student_id}/metadata/"
    )
    metadata = {
        "student_id":student_id, # 
        "current_status":student_row['current_status'], #
        "education_level":student_row['education_level'], #
        "education_major":student_row['education_major'], #
        "target_roles":student_row['target_roles'], #
        "timezone":cfg["app"]["timezone"], #
        "model_name":cfg["llm"]["model_name"], #
        "max_output_tokens":cfg["llm"]["max_output_tokens"], #
        "feed_text_max_chars":cfg["hyde"]["feed_text_max_chars"], #
        "temperature":cfg["llm"]["temperature"] #
    }
    print(metadata)
    cgs.upload_json(
        blob_path   = f"{student_id}/metadata/metadata.json",
        json_data   = metadata
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding01.npy",
        array       = emb[0]
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding02.npy",
        array       = emb[1]
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding03.npy",
        array       = emb[2]
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding04.npy",
        array       = emb[3]
    )
    cgs.upload_npy(
        blob_path   = f"{student_id}/embedding/embedding05.npy",
        array       = emb[4]
    )
    cgs.upload_text(
        blob_path = f"{student_id}/hyde/hyde_text01.txt",
        text_data = hyde_query_texts[0]
    )
    cgs.upload_text(
        blob_path = f"{student_id}/hyde/hyde_text02.txt",
        text_data = hyde_query_texts[1]
    )
    cgs.upload_text(
        blob_path = f"{student_id}/hyde/hyde_text03.txt",
        text_data = hyde_query_texts[2]
    )
    cgs.upload_text(
        blob_path = f"{student_id}/hyde/hyde_text04.txt",
        text_data = hyde_query_texts[3]
    )
    cgs.upload_text(
        blob_path = f"{student_id}/hyde/hyde_text05.txt",
        text_data = hyde_query_texts[4]
    )
    
    # break

2026-02-08T18:14:43Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=8.763s | in_tokens=816 | out_tokens=332 | model=gemini-2.5-flash | status=ok
2026-02-08T18:14:45Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p007 events=4 prompt=hyde_a


####################################################################################################
emb.shape -> (5, 768)
**************************************************
(5, 768)
0
student_row -> 
 {'student_id': 'stu_p007', 'preferred_language': 'th', 'current_status': 'newgrad', 'education_level': 'bachelor', 'education_major': 'วิศวกรรมอุตสาหการ', 'target_roles': 'Quality Engineer', 'skills': 'Statistics:L1;Documentation:L1', 'interests': 'เตรียมสัมภาษณ์;หางาน', 'onboard_grp': 'Job_Hunter', 'onboard_grp_description': 'เพิ่งจบและหางานสายคุณภาพ/กระบวนการ'}
user_ctx -> 
 UserContextArtifacts(user_context_json={'student_id': 'stu_p007', 'preferred_language': 'th', 'current_status': 'newgrad', 'education': {'level': 'bachelor', 'major': 'วิศวกรรมอุตสาหการ'}, 'target_roles': [{'role_id': 'quality_engineer', 'role_name': 'Quality Engineer', 'priority': 1}], 'skills': [{'skill_id': 'statistics', 'skill_name': 'Statistics', 'proficiency': 'L1'}, {'skill_id': 'documentation', 'skill_name'

2026-02-08T18:14:55Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=7.121s | in_tokens=965 | out_tokens=340 | model=gemini-2.5-flash | status=ok
2026-02-08T18:14:57Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p006 events=6 prompt=hyde_b


####################################################################################################
emb.shape -> (5, 768)
**************************************************
(5, 768)
1
student_row -> 
 {'student_id': 'stu_p006', 'preferred_language': 'th', 'current_status': 'student2yr', 'education_level': 'bachelor', 'education_major': 'ชีววิทยา', 'target_roles': 'Biotechnology Intern', 'skills': 'Biology Fundamentals:L1;Lab Skills:unknown', 'interests': 'ฝึกงาน;ทำพอร์ตสาย Bio;สมัครทุน', 'onboard_grp': 'Job_Hunter', 'onboard_grp_description': 'อยากได้ฝึกงานสายชีวภาพและอยากเตรียมพอร์ต'}
user_ctx -> 
 UserContextArtifacts(user_context_json={'student_id': 'stu_p006', 'preferred_language': 'th', 'current_status': 'student2yr', 'education': {'level': 'bachelor', 'major': 'ชีววิทยา'}, 'target_roles': [{'role_id': 'biotechnology_intern', 'role_name': 'Biotechnology Intern', 'priority': 1}], 'skills': [{'skill_id': 'biology_fundamentals', 'skill_name': 'Biology Fundamentals', 'proficiency': '

2026-02-08T18:15:08Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=8.919s | in_tokens=890 | out_tokens=331 | model=gemini-2.5-flash | status=ok
2026-02-08T18:15:11Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p001 events=9 prompt=hyde_b


...

####################################################################################################
emb.shape -> (5, 768)
**************************************************
(5, 768)
8
student_row -> 
 {'student_id': 'stu_p010', 'preferred_language': 'th', 'current_status': 'student3yr', 'education_level': 'bachelor', 'education_major': 'ความสัมพันธ์ระหว่างประเทศ', 'target_roles': 'Policy Analyst', 'skills': 'Writing:L1;Research:L1', 'interests': 'สมัครทุน;ข่าวมหาวิทยาลัย;เส้นทางอาชีพ', 'onboard_grp': 'Learner', 'onboard_grp_description': 'สนใจทุนและเส้นทางอาชีพด้านนโยบาย (harder match)'}
user_ctx -> 
 UserContextArtifacts(user_context_json={'student_id': 'stu_p010', 'preferred_language': 'th', 'current_status': 'student3yr', 'education': {'level': 'bachelor', 'major': 'ความสัมพันธ์ระหว่างประเทศ'}, 'target_roles': [{'role_id': 'policy_analyst', 'role_name': 'Policy Analyst', 'priority': 1}], 'skills': [{'skill_id': 'writing', 'skill_name': 'Writing', 'proficiency': 'L1'}, {'skill_id': 

2026-02-08T18:17:11Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=6.520s | in_tokens=910 | out_tokens=319 | model=gemini-2.5-flash | status=ok
2026-02-08T18:17:13Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p008 events=5 prompt=hyde_b


####################################################################################################
emb.shape -> (5, 768)
**************************************************
(5, 768)
9
student_row -> 
 {'student_id': 'stu_p008', 'preferred_language': 'th', 'current_status': 'student0yr', 'education_level': 'bachelor', 'education_major': 'นิเทศศาสตร์', 'target_roles': 'Undecided', 'skills': 'Communication:L1', 'interests': 'ข่าวมหาวิทยาลัย;กิจกรรม;ทุน', 'onboard_grp': 'Social', 'onboard_grp_description': 'นักศึกษาใหม่ ต้องการข่าวสารมหาวิทยาลัยและโอกาสทุน'}
user_ctx -> 
 UserContextArtifacts(user_context_json={'student_id': 'stu_p008', 'preferred_language': 'th', 'current_status': 'student0yr', 'education': {'level': 'bachelor', 'major': 'นิเทศศาสตร์'}, 'target_roles': [{'role_id': 'undecided', 'role_name': 'Undecided', 'priority': 1}], 'skills': [{'skill_id': 'communication', 'skill_name': 'Communication', 'proficiency': 'L1'}], 'interests': ['ข่าวมหาวิทยาลัย', 'กิจกรรม', 'ทุน'], 'onboa

2026-02-08T18:17:22Z | INFO | functions.utils.llm_client | LLM call done | attempt=1 | latency=7.724s | in_tokens=782 | out_tokens=302 | model=gemini-2.5-flash | status=ok
2026-02-08T18:17:25Z | INFO | pipeline_1_user_hyde | wrote HyDE bundle student_id=stu_p000 events=4 prompt=hyde_a


####################################################################################################
emb.shape -> (5, 768)
**************************************************
(5, 768)
10
student_row -> 
 {'student_id': 'stu_p000', 'preferred_language': 'en', 'current_status': 'student', 'education_level': 'bachelor', 'education_major': 'electrical engineering', 'target_roles': 'data science', 'skills': 'python;sql;statistics', 'interests': 'machine learning;career growth', 'onboard_grp': 'job_hunter', 'onboard_grp_description': 'looking to transition into data science role'}
user_ctx -> 
 UserContextArtifacts(user_context_json={'student_id': 'stu_p000', 'preferred_language': 'en', 'current_status': 'student', 'education': {'level': 'bachelor', 'major': 'electrical engineering'}, 'target_roles': [{'role_id': 'data_science', 'role_name': 'data science', 'priority': 1}], 'skills': [{'skill_id': 'python', 'skill_name': 'python', 'proficiency': 'unknown'}, {'skill_id': 'sql', 'skill_name': '